In [10]:
from huggingface_hub import whoami

try:
    user_info = whoami()
    print(f"Logged in as: {user_info['name']}")
    print(f"Full Name: {user_info.get('fullname', 'N/A')}")
except Exception as e:
    print("Not logged in or token is invalid.")

Logged in as: mia-n
Full Name: Mia Natali


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

MODEL_ID = "RichardLu/Llama3_AE_laptop"

# Use Apple Metal when available, else CPU
device = "mps" if torch.backends.mps.is_available() else "cpu"
dtype = torch.float16 if device == "mps" else torch.float32

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=dtype)
model.to(device)
model.eval()

# Define the instruction for aspect extraction
instructabsa_instruction = """Definition: The output will be the aspects (both implicit and explicit) which have an associated opinion that are extracted from the input text. In cases where there are no aspects the output should be noaspectterm.
Positive example 1-
input: The datacenters bring lots of jobs to our community and I kind of like the way that the buildings look.
output: jobs, visual impact
Positive example 2-
input: The datacenters near my home ensure that my power and water won't run out, because they need a constant supply. I've also never heard any noise from the generators and don't have any problems with the light.
output: utilities, power, water, quality of life
Negative example 1-
input: They are so loud and so bright that I'm not able to sleep at night. Everyone on my street has the same problem. This is making housing prices go down so we can't even sell at a good rate anymore.
output: noise, light pollution, quality of life, housing
Negative example 2-
input: I don't understand why the mayor won't do anything to stop the spread of these datacenters when everybody has been complaining about them nonstop.
output: government
Neutral example 1-
input: I heard about the effects of datacenters on the economy.
output: economy
Neutral example 2-
input: I need to read more about the effects of datacenters on climate change and wildlife.
output: environment, climate change, wildlife
Now complete the following example:"""

alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.
### Instruction:
{}
### Input:
{}
### Response:
# {}"""

prompt = alpaca_prompt.format(instructabsa_instruction, "While I am not opposed to data centers, what I am opposed to is that they keep getting build in areas that need more housing and should be zoned for residential or mixed residential. There are issues with home affordability and the only way out of it is to build new housing. Additionally, as these data center projects are completed I worry about the demand on our power grid.", "")

inputs = tokenizer(prompt, return_tensors="pt").to(device)
with torch.no_grad():
    output_ids = model.generate(**inputs, max_new_tokens=128, do_sample=False)

output_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
print(output_text.split("### Response:")[-1].strip())
